# 09a — Memory and serving performance — Practice

Read the lesson and run the example. Implement each TODO, then run its checks. A NotImplementedError is expected until you write your answer. Try for 20–30 minutes before opening a hint; open the solution only after recording an attempt. Copy this notebook into work/ to keep your own version.

**Prerequisites:** earlier lessons in the course order. **Environment:** Python standard library only; Jupyter is needed for the notebook UI.

[Stage guide](../stages/09.md) · [Course map](../PLAN.md) · [Project](../../projects/stage09/README.md)

**Notebook time:** 2–4 hours for an initial pass; project, reading, retrieval practice, and independent transfer use the rest of the stage budget.

Learning objectives:

- Estimate weight and KV memory
- Distinguish latency from throughput
- Use measurements to choose serving settings


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets" / "financials.csv").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the learning repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def expect_error(error_type, operation):
    try:
        operation()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

print("Course root:", ROOT)


## Lesson

Weight memory is roughly parameter count × bits per parameter / 8. This estimate excludes quantization metadata, activations, temporary workspaces, runtime overhead, and KV cache. FP16 and BF16 both use two bytes but have different numerical ranges. CUDA is NVIDIA’s compute platform; other devices use other backends. A model fitting in RAM does not imply acceptable latency.

For a decoder, KV cache storage roughly scales with 2 × layers × KV heads × head dimension × sequence length × batch × bytes. Grouped-query attention can reduce KV heads relative to query heads. Longer contexts and more concurrent users can dominate memory.

Latency measures a request’s elapsed time; throughput measures completed work per unit time. Report time to first token, inter-token latency, tokens/second, concurrency, prompt/output lengths, and p50/p95. Batching amortizes work but can increase waiting time. Continuous batching allows sequences to enter/leave a running batch. Do not select hardware from weight size alone.


## Worked example

Predict the output before running. Change one input and explain the result.


In [ ]:
print("7 billion FP16 weights, GiB:", 7_000_000_000*2/1024**3)


## Exercise 1: Weight estimate

Return GiB for parameters and bits; require positive values.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def weight_gib(parameters, bits):
    raise NotImplementedError


In [ ]:
assert weight_gib(1024**3, 8) == 1
assert weight_gib(1024**3, 4) == .5
print("Exercise 1: checks passed")


<details><summary>Hint — open after trying</summary>

GiB uses powers of 1024.

</details>


## Exercise 2: KV estimate

Return bytes for layers, kv_heads, head_dim, tokens, batch=1, bytes_per_value=2.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def kv_bytes(layers, kv_heads, head_dim, tokens, batch=1, bytes_per_value=2):
    raise NotImplementedError


In [ ]:
assert kv_bytes(2, 4, 8, 16) == 4096
print("Exercise 2: checks passed")


<details><summary>Hint — open after trying</summary>

The leading 2 counts keys and values.

</details>


## Exercise 3: Nearest-rank percentile

Return nearest-rank percentile p in (0,1] of nonempty values.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def percentile(values, p):
    raise NotImplementedError


In [ ]:
assert percentile(list(range(1, 101)), .95) == 95
assert percentile([3], .5) == 3
print("Exercise 3: checks passed")


<details><summary>Hint — open after trying</summary>

Sort then select ceil(p*n)-1.

</details>


## Independent transfer

Create a hardware budget for 1B, 7B, and 14B models at 4/8/16 bits and multiple context lengths. Measure your actual machine before buying hardware or choosing a deployment target.

Record your implementation, errors, measurements, and explanation in work/. See the project’s ANSWERS.md after attempting the brief.


## Explain without code

1. Why does concurrency consume memory?

2. Is quantization always faster?


## Reading and review

- [Primary reference 1](https://docs.vllm.ai/en/latest/)

After 2, 7, and 30 days: explain the concept from memory, solve a changed-input version, and log any gaps in progress.md.
